In [21]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix
from numpy.linalg import solve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [22]:
# === Initialization ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

case_name = 'pglib_opf_case3_lmbd'
case_path = f'../excel_outputs/{case_name}.xlsx'
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

bus_to_idx = {bus: i+1 for i, bus in enumerate(case['bus'].bus_i.values)}
bus_idx = [bus_to_idx[bus] for bus in case['bus'].bus_i.values]
case['bus'].bus_i = case['bus'].bus_i.replace(bus_to_idx) # rename the bus for making PTDF
case['gen'].bus_i = case['gen'].bus_i.replace(bus_to_idx)
case['gencost'].bus_i = case['gencost'].bus_i.replace(bus_to_idx)
case['branch'].bus_i = case['branch'].bus_i.replace(bus_to_idx)
case['branch'].bus_j = case['branch'].bus_j.replace(bus_to_idx)
nbus = case['bus'].shape[0]
ngen = case['gen'].shape[0]
nbranch = case['branch'].shape[0]

# per unit p.u. conversion for cost coefficients
baseMVA = case['baseMVA'].values[0][0]
c2 = case['gencost'].c2.values * baseMVA**2
c1 = case['gencost'].c1.values * baseMVA
c0 = case['gencost'].c0.values

# calculate susceptance, conductance, admittance-square y_sq
# $Z = r + ix$ $Y = g + ib$ $Y = \frac{1}{Z} = \frac{r}{r^2 + x^2} - i\frac{x}{r^2 + x^2}$
# 1. Physics: Admittance Y = g + i*b
r = case['branch']['r'].values
x = case['branch']['x'].values
Z_sq = r**2 + x**2
g = r / Z_sq
b = -x / Z_sq
y_sq = 1 / Z_sq

# 2. Extract Line Charging, Taps, and Phase Shifts
bc = case['branch']['b'].values # MATPOWER branch 'b' is total line charging susceptance
tau = np.where(case['branch']['ratio'].values == 0, 1.0, case['branch']['ratio'].values)
theta_shift = np.radians(case['branch']['angle'].values)

# 3. Data Extraction
Gs = case['bus']['Gs'].values / baseMVA
Bs = case['bus']['Bs'].values / baseMVA
Pd = case['bus'].Pd.values / baseMVA
Qd = case['bus'].Qd.values / baseMVA

# State vector dimension D = 2 * |B|
D = 2 * nbus

# Data for QCQP

## 1. Branch Flow Matrices (Equations 14-22)

In [ ]:
# Initialize lists to store matrices for all branches
M_pf = []; M_qf = []; M_pt = []; M_qt = []

# Pre-calculate derived branch elements
g11 = g / (tau**2)
g12 = g * np.cos(theta_shift) / tau
g21 = g * np.sin(theta_shift) / tau
g22 = g

b11 = (b + bc/2) / (tau**2)
b12 = b * np.cos(theta_shift) / tau
b21 = b * np.sin(theta_shift) / tau
b22 = b + bc/2

for l in range(nbranch):
    # Python uses 0-based indexing; your dictionary offset to 1-based, so we subtract 1
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    
    # Real and Imaginary indices
    i_B = i + nbus
    j_B = j + nbus

    # --- FROM END ---
    A_pf = np.zeros((D, D))
    A_pf[i, i] = g11[l]
    A_pf[i_B, i_B] = g11[l]
    A_pf[i, j] = -(g12[l] - b21[l])
    A_pf[i_B, j_B] = -(g12[l] - b21[l])
    A_pf[i, j_B] = (g21[l] + b12[l])
    A_pf[i_B, j] = -(g21[l] + b12[l])
    M_pf.append(0.5 * (A_pf + A_pf.T))

    A_qf = np.zeros((D, D))
    A_qf[i, i] = -b11[l]
    A_qf[i_B, i_B] = -b11[l]
    A_qf[i, j] = (b12[l] + g21[l])
    A_qf[i_B, j_B] = (b12[l] + g21[l])
    A_qf[i, j_B] = -(b21[l] - g12[l])
    A_qf[i_B, j] = (b21[l] - g12[l])
    M_qf.append(0.5 * (A_qf + A_qf.T))

    # --- TO END ---
    A_pt = np.zeros((D, D))
    A_pt[j, j] = g22[l]
    A_pt[j_B, j_B] = g22[l]
    A_pt[j, i] = -(g12[l] + b21[l])
    A_pt[j_B, i_B] = -(g12[l] + b21[l])
    A_pt[j, i_B] = -(g21[l] - b12[l])
    A_pt[j_B, i] = (g21[l] - b12[l])
    M_pt.append(0.5 * (A_pt + A_pt.T))

    A_qt = np.zeros((D, D))
    A_qt[j, j] = -b22[l]
    A_qt[j_B, j_B] = -b22[l]
    A_qt[j, i] = (b12[l] - g21[l])
    A_qt[j_B, i_B] = (b12[l] - g21[l])
    A_qt[j, i_B] = (b21[l] + g12[l])
    A_qt[j_B, i] = -(b21[l] + g12[l])
    M_qt.append(0.5 * (A_qt + A_qt.T))
    

## 2. Nodal Power Injection Matrices (Equations 23-28)

In [24]:
# 1. Build Standard Y_bus matrices (G_bus and B_bus)
G_bus = np.zeros((nbus, nbus))
B_bus = np.zeros((nbus, nbus))

# Add shunts to the diagonals
np.fill_diagonal(G_bus, Gs)
np.fill_diagonal(B_bus, Bs)

# Add branch contributions
for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1

    # Diagonal elements
    G_bus[i, i] += g11[l]
    G_bus[j, j] += g22[l]
    B_bus[i, i] += b11[l]
    B_bus[j, j] += b22[l]

    # Off-diagonal elements (symmetric for the line itself, but tap ratios make Y_bus asymmetric)
    G_bus[i, j] += -g12[l]
    G_bus[j, i] += -g12[l] # Assuming simplified model where g12 roughly handles the tap phase
    B_bus[i, j] += -b12[l]
    B_bus[j, i] += -b12[l]

# 2. Build the QCQP Nodal Matrices
M_p = []; M_q = []

for i in range(nbus):
    A_pi = np.zeros((D, D))
    A_qi = np.zeros((D, D))
    
    # Active Power Row mappings (Eq 25)
    A_pi[i, :nbus] = G_bus[i, :]
    A_pi[i, nbus:] = -B_bus[i, :]
    A_pi[i + nbus, :nbus] = B_bus[i, :]
    A_pi[i + nbus, nbus:] = G_bus[i, :]
    M_p.append(0.5 * (A_pi + A_pi.T))
    
    # Reactive Power Row mappings (Eq 27)
    A_qi[i, :nbus] = -B_bus[i, :]
    A_qi[i, nbus:] = -G_bus[i, :]
    A_qi[i + nbus, :nbus] = G_bus[i, :]
    A_qi[i + nbus, nbus:] = -B_bus[i, :]
    M_q.append(0.5 * (A_qi + A_qi.T))

# 3. Build the QCQP current limit matrices M_I
M_I = []
for l in range(nbranch):
    # Extract 0-based indices directly
    i = int(case['branch']['bus_i'].values[l]) - 1 
    j = int(case['branch']['bus_j'].values[l]) - 1
    
    i_B = i + nbus
    j_B = j + nbus

    M = np.zeros((D, D))
    
    # Real part block
    M[i, i] += y_sq[l]
    M[j, j] += y_sq[l]
    M[i, j] -= y_sq[l]
    M[j, i] -= y_sq[l]

    # Imaginary part block
    M[i_B, i_B] += y_sq[l]
    M[j_B, j_B] += y_sq[l]
    M[i_B, j_B] -= y_sq[l]
    M[j_B, i_B] -= y_sq[l]
    
    M_I.append(M)

# 2. Tensor Stack Generation
M_I_stack = torch.stack([
    torch.as_tensor(M_I[l], dtype=dtype, device=device) 
    for l in range(nbranch)
])


## 3. Branch Angle Difference & Voltage Matrices (Equations 29-34)

In [25]:
M_c = []; M_s = []

for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    i_B = i + nbus
    j_B = j + nbus

    # Angle Cosine Extraction (Eq 29 & 30)
    A_c = np.zeros((D, D))
    A_c[i, j] = 1
    A_c[i_B, j_B] = 1
    M_c.append(0.5 * (A_c + A_c.T))

    # Angle Sine Extraction (Eq 31 & 32)
    A_s = np.zeros((D, D))
    A_s[i_B, j] = 1
    A_s[i, j_B] = -1
    M_s.append(0.5 * (A_s + A_s.T))

M_V = []
for i in range(nbus):
    # Voltage Magnitude Extraction (Eq 33 & 34)
    A_V = np.zeros((D, D))
    A_V[i, i] = 1
    A_V[i + nbus, i + nbus] = 1
    M_V.append(A_V) # Already symmetric

## 4. Reference Angle Vector

In [26]:
# Identify the slack bus (MATPOWER sets bus type to 3 for slack)
slack_bus_idx = case['bus'][case['bus']['type'] == 3].index[0]

a_ref = np.zeros(D)
# Force the imaginary voltage component of the slack bus to 0
a_ref[slack_bus_idx + nbus] = 1

# Torch stack

## Tensor prep

In [ ]:
# ------------------------------------------------------------
# 1) Dimensions
# ------------------------------------------------------------
# These match the sizes from your MATPOWER data
nbus = nbus
ngen = ngen
nbranch = nbranch
d = D

# ------------------------------------------------------------
# 2) Stack quadratic matrices (For ALL buses and branches)
# ------------------------------------------------------------
# Nodal power and voltage matrices [nbus, d, d]
M_p_stack = torch.stack([torch.as_tensor(M_p[i], dtype=dtype, device=device) for i in range(nbus)])
M_q_stack = torch.stack([torch.as_tensor(M_q[i], dtype=dtype, device=device) for i in range(nbus)])
M_v_stack = torch.stack([torch.as_tensor(M_V[i], dtype=dtype, device=device) for i in range(nbus)])

# Angle difference matrices [nbranch, d, d]
M_c_stack = torch.stack([torch.as_tensor(M_c[l], dtype=dtype, device=device) for l in range(nbranch)])
M_s_stack = torch.stack([torch.as_tensor(M_s[l], dtype=dtype, device=device) for l in range(nbranch)])

# ------------------------------------------------------------
# 3) Symmetrize matrices (Required for stable Autograd)
# ------------------------------------------------------------
M_p_stack = 0.5 * (M_p_stack + M_p_stack.transpose(-1, -2))
M_q_stack = 0.5 * (M_q_stack + M_q_stack.transpose(-1, -2))
M_v_stack = 0.5 * (M_v_stack + M_v_stack.transpose(-1, -2))

M_c_stack = 0.5 * (M_c_stack + M_c_stack.transpose(-1, -2))
M_s_stack = 0.5 * (M_s_stack + M_s_stack.transpose(-1, -2))

# ------------------------------------------------------------
# 4) The C_g Matrix (Mapping Generators to Buses)
# ------------------------------------------------------------
# Shape: [nbus, ngen]
C_g = torch.zeros((nbus, ngen), dtype=dtype, device=device)
for gen_idx, bus_i in enumerate(case['gen']['bus_i'].values):
    bus_idx = int(bus_i) - 1 # convert to 0-based index
    C_g[bus_idx, gen_idx] = 1.0

# ------------------------------------------------------------
# 5) Vectors: Demands, Limits, and Reference
# ------------------------------------------------------------
Pd_bus = np.asarray(case['bus'].Pd.values, dtype=np.float32) / baseMVA
Qd_bus = np.asarray(case['bus'].Qd.values, dtype=np.float32) / baseMVA

pmax = np.asarray(case['gen'].Pmax.values, dtype=np.float32) / baseMVA
pmin = np.asarray(case['gen'].Pmin.values, dtype=np.float32) / baseMVA
qmax = np.asarray(case['gen'].Qmax.values, dtype=np.float32) / baseMVA
qmin = np.asarray(case['gen'].Qmin.values, dtype=np.float32) / baseMVA

# current branch limits (Imax)
Imax = np.asarray(case['branch'].rateA.values, dtype=np.float32) / baseMVA
Imax[Imax == 0] = 9999.0  # Handle unconstrained lines gracefully

# Branch angle limits (converted to radians)
angmax = np.radians(np.asarray(case['branch'].angmax.values, dtype=np.float32))
angmin = np.radians(np.asarray(case['branch'].angmin.values, dtype=np.float32))

Vmax_arr = np.asarray(case['bus'].Vmax.values, dtype=np.float32)
Vmin_arr = np.asarray(case['bus'].Vmin.values, dtype=np.float32)

# ------------------------------------------------------------
# 6) Final problem dictionary for the PINN loss
# ------------------------------------------------------------
problem = {
    # Quadratic Matrices
    "M_p": M_p_stack,
    "M_q": M_q_stack,
    "M_v": M_v_stack,
    "M_I": M_I_stack,
    "M_c": M_c_stack,
    "M_s": M_s_stack,
    
    # Incidence Matrix
    "C_g": C_g,

    # Base Vectors
    "Pd": torch.as_tensor(Pd_bus, dtype=dtype, device=device),
    "Qd": torch.as_tensor(Qd_bus, dtype=dtype, device=device),
    
    "pmax": torch.as_tensor(pmax, dtype=dtype, device=device),
    "pmin": torch.as_tensor(pmin, dtype=dtype, device=device),
    "qmax": torch.as_tensor(qmax, dtype=dtype, device=device),
    "qmin": torch.as_tensor(qmin, dtype=dtype, device=device),
    
    "Imax": torch.as_tensor(Imax, dtype=dtype, device=device),
    "angmax": torch.as_tensor(angmax, dtype=dtype, device=device),
    "angmin": torch.as_tensor(angmin, dtype=dtype, device=device),
    
    # the cost coefficients
    "c2": torch.as_tensor(c2, dtype=dtype, device=device),
    "c1": torch.as_tensor(c1, dtype=dtype, device=device),
    "c0": torch.as_tensor(c0, dtype=dtype, device=device),

    "Vmax": torch.as_tensor(Vmax_arr, dtype=dtype, device=device),
    "Vmin": torch.as_tensor(Vmin_arr, dtype=dtype, device=device),
    
    # Anchor vector (Ensure a_ref from our earlier discussion is defined)
    "a_ref": torch.as_tensor(a_ref, dtype=dtype, device=device),

    # Metadata
    "nbus": nbus,
    "ngen": ngen,
    "nbranch": nbranch
}

print("Constructed PINN problem data for QCQP:")
print(f"  nbus    = {nbus}")
print(f"  ngen    = {ngen}")
print(f"  nbranch = {nbranch}")
print(f"  C_g shape  = {tuple(problem['C_g'].shape)}")
print(f"  M_p, M_q shape  = {tuple(problem['M_p'].shape)}, {tuple(problem['M_q'].shape)}")
print(f"  M_s, M_c shape  = {tuple(problem['M_s'].shape)}, {tuple(problem['M_c'].shape)}")
print(f"  M_V shape  = {tuple(problem['M_v'].shape)}")

Constructed PINN problem data for QCQP:
  nbus    = 3
  ngen    = 3
  nbranch = 3
  C_g shape  = (3, 3)
  M_p, M_q shape  = (3, 6, 6), (3, 6, 6)
  M_s, M_c shape  = (3, 6, 6), (3, 6, 6)
  M_V shape  = (3, 6, 6)


# Random demand

In [35]:
def gaussian_batch(base_tensor, batch_size, variation_std=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with Gaussian random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation_std (float): standard deviation of Gaussian noise, relative to base_tensor
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    # Expand base tensor to batch
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Gaussian noise scaled by base_tensor
    noise = variation_std * base_tensor.unsqueeze(0) * torch.randn_like(base_batch)
    
    # Perturbed batch
    batch = base_batch + noise
    
    # Clamp if necessary
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch

def uniform_batch(base_tensor, batch_size, variation=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with uniform random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation (float): maximum relative deviation (±variation)
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Uniform noise in [-variation, +variation]
    noise = (2 * torch.rand_like(base_batch) - 1) * variation
    
    batch = base_batch * (1 + noise)
    
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch


# Base line model

## 1. PINN Architecture

In [36]:
class FeasibleQCQPMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]
    Output:
        v:  [B, 2*nbus] (Rectangular voltages)
        pg: [B, ngen]   (Active generation)
        qg: [B, ngen]   (Reactive generation)
    """
    def __init__(self, nbus: int, ngen: int, slack_imag_idx: int, hidden: int = 512):
        super().__init__()
        self.nbus = nbus
        self.ngen = ngen
        self.in_dim = 2 * nbus
        self.out_dim_v = 2 * nbus
        self.out_dim_g = 2 * ngen 
        self.slack_imag_idx = int(slack_imag_idx)

        # Core MLP
        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, self.out_dim_v + self.out_dim_g),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, problem: dict) -> tuple:
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)
        raw = self.net(x)

        # 1. Slice outputs
        v_raw = raw[:, :self.out_dim_v]
        g_raw = raw[:, self.out_dim_v:]
        
        pg_raw = g_raw[:, :self.ngen]
        qg_raw = g_raw[:, self.ngen:]

        # 2. Bound Voltages to [-Vmax, Vmax] using Tanh for smooth gradients
        Vmax_b = problem["Vmax"].reshape(1, -1).expand(B, -1)
        Vmax_full = torch.cat([Vmax_b, Vmax_b], dim=-1) # For real and imaginary parts
        v = torch.tanh(v_raw) * Vmax_full

        # Constraint (2m): Enforce slack imaginary part = 0 exactly
        v_clone = v.clone()
        v_clone[:, self.slack_imag_idx] = 0.0
        v = v_clone

        # 3. Bound Generation strictly between [min, max] using Sigmoid
        pmax_b = problem["pmax"].reshape(1, -1).expand(B, -1)
        pmin_b = problem["pmin"].reshape(1, -1).expand(B, -1)
        qmax_b = problem["qmax"].reshape(1, -1).expand(B, -1)
        qmin_b = problem["qmin"].reshape(1, -1).expand(B, -1)

        pg = pmin_b + torch.sigmoid(pg_raw) * (pmax_b - pmin_b)
        qg = qmin_b + torch.sigmoid(qg_raw) * (qmax_b - qmin_b)

        return v, pg, qg

## 2. The QCQP Loss Function

In [ ]:
def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_qcqp_loss(model: nn.Module, Pd_batch: torch.Tensor, Qd_batch: torch.Tensor, problem: dict, weights: dict):
    B = Pd_batch.shape[0]
    
    v, pg, qg = model(Pd_batch, Qd_batch, problem)

    M_p, M_q = problem["M_p"], problem["M_q"]
    M_I = problem["M_I"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    # Expand limits for batch
    Imax = problem["Imax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    
    # Expand Cost Coefficients
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1)

    # --------------------------------------------------------
    # STRICTLY QUADRATIC EVALUATIONS (O(v^2))
    # --------------------------------------------------------
    vp = quad_batch_stack(v, M_p)   
    vq = quad_batch_stack(v, M_q)
    vI = quad_batch_stack(v, M_I)
    vc = quad_batch_stack(v, M_c)
    vs = quad_batch_stack(v, M_s)
    vv = quad_batch_stack(v, M_v)

    # --------------------------------------------------------
    # Constraints Mapping
    # --------------------------------------------------------
    # 1. Branch Current Limit (Eq 2b)
    g_I = vI - (Imax**2)
    
    # 2. Nodal Power Balance (Eq 2c & 2d)
    h_p = (pg @ C_g.T) - Pd_batch - vp
    h_q = (qg @ C_g.T) - Qd_batch - vq

    # 3. Angle Difference (Eq 2e & 2f)
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc

    # 4. Voltage Magnitude Security (Eq 2g)
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv

    # 5. Objective (Eq 2a) - Minimize Quadratic Generation Cost
    # Element-wise operations equivalent to c2*(pg^2) + c1*pg + c0
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()

    # --------------------------------------------------------
    # Compute Penalties
    # --------------------------------------------------------
    loss_eq_p = h_p.pow(2).mean()
    loss_eq_q = h_q.pow(2).mean()
    loss_thermal = F.relu(g_I).pow(2).mean()
    loss_ang = F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean()
    loss_v = F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean()

    total_loss = (
        weights["eq_p"] * loss_eq_p +
        weights["eq_q"] * loss_eq_q +
        weights["thermal"] * loss_thermal +
        weights["ang"] * loss_ang +
        weights["v"] * loss_v +
        weights["obj"] * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_eq_p": loss_eq_p.detach().item(),
        "max_h_p_viol": h_p.abs().max().detach().item(),
        "max_thermal_viol": F.relu(g_I).max().detach().item(),
        "objective_cost": obj.detach().item()
    }

    return total_loss, diagnostics

## 3. Training

In [31]:
# Extract the exact index where a_ref is 1 (the slack bus imaginary component)
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model = FeasibleQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Define penalty weights (tune these based on convergence)
loss_weights = {
    "eq_p": 10.0,
    "eq_q": 10.0,
    "thermal": 5.0,
    "ang": 1.0,
    "v": 5.0,
    "obj": 0.01
}

# Example Training Loop
for epoch in range(1000):
    # Base demands from problem dict
    Pd_base = problem["Pd"]
    Qd_base = problem["Qd"]
    
    # Generate random scenario batch
    Pd_batch = gaussian_batch(Pd_base, batch_size=32)
    Qd_batch = gaussian_batch(Qd_base, batch_size=32)
    
    optimizer.zero_grad()
    loss, diag = compute_qcqp_loss(model, Pd_batch, Qd_batch, problem, loss_weights)
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {diag['loss_total']:.4f} | Max P-Balance Error: {diag['max_h_p_viol']:.4f}")

Epoch    0 | Loss: 569.8160 | Max P-Balance Error: 9.4605
Epoch  100 | Loss: 2.3441 | Max P-Balance Error: 0.6284
Epoch  200 | Loss: 0.1145 | Max P-Balance Error: 0.2551
Epoch  300 | Loss: 0.1512 | Max P-Balance Error: 0.1209
Epoch  400 | Loss: 0.0543 | Max P-Balance Error: 0.1181
Epoch  500 | Loss: 0.1345 | Max P-Balance Error: 0.2299
Epoch  600 | Loss: 0.1109 | Max P-Balance Error: 0.1173
Epoch  700 | Loss: 0.0828 | Max P-Balance Error: 0.0964
Epoch  800 | Loss: 0.0808 | Max P-Balance Error: 0.0978
Epoch  900 | Loss: 0.0705 | Max P-Balance Error: 0.0925


# Old